
### Structured output
Models can be requested to provide their response in a format matching a given schema. This is useful for ensuring the output can be easily parsed and used in subsequent processing. LangChain supports multiple schema types and methods for enforcing structured output.

### Pydantic
Pydantic models provide the richest feature set with field validation, descriptions, and nested structures.

In [3]:
import os 
from dotenv import load_dotenv
load_dotenv()
from langchain.chat_models import init_chat_model
os.environ["GROQ_API_KEY"] = os.getenv("GROQ_API_KEY")

model = init_chat_model("groq:openai/gpt-oss-20b")

In [4]:
from pydantic import BaseModel, Field 
class Movie(BaseModel):
    title:str=Field(description="The title of the movie")
    year:int=Field(description="The year the movie was released")
    director:str=Field(description="The director of the movie")
    rating:float=Field(description="The movie rating out of 10")
    review:str=Field(description="A review of the movie")
    plot:str=Field(description="The plot of the movie")
    cast:list[str]=Field(description="The cast of the movie")
    genre:list[str]=Field(description="The genre of the movie")


In [5]:
model_with_structured_output = model.with_structured_output(Movie)

In [6]:
model_with_structured_output

_ChatModelBinding(bound=ChatGroq(metadata={'lc_versions': {'langchain-core': '1.6.6', 'langchain': '1.4.3'}}, profile={'name': 'GPT OSS 20B', 'release_date': '2025-08-05', 'last_updated': '2026-05-27', 'open_weights': True, 'max_input_tokens': 131072, 'max_output_tokens': 65536, 'text_inputs': True, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'text_outputs': True, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': True, 'tool_calling': True, 'structured_output': True, 'attachment': False, 'temperature': True}, client=<groq.resources.chat.completions.Completions object at 0x00000250747098B0>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x000002507485A150>, model_name='openai/gpt-oss-20b', model_kwargs={}, groq_api_key=SecretStr('**********')), kwargs={'tools': [{'type': 'function', 'function': {'name': 'Movie', 'description': '', 'parameters': {'properties': {'title': {'description': 'The titl

In [7]:
model.invoke("Proved details about the movie Inception")

AIMessage(content='**Inception** (2010) – A Comprehensive, Fact‑Verified Overview  \n\n| Item | Detail |\n|------|--------|\n| **Title** | *Inception* |\n| **Release Date** | **July 16, 2010** (USA) – 1:30\u202fp.m. (Pacific Time) |\n| **Director / Writer** | **Christopher\u202fNolan** (both roles) |\n| **Producer** | Christopher Nolan, Emma Thomas, Emma Thomas, and others (via Syncopy and Warner Bros.) |\n| **Production Companies** | Syncopy Inc., Warner Bros. Pictures |\n| **Cinematography** | Wally Pfister (Academy Award‑winning) |\n| **Music** | Hans Zimmer (notably the “braaam” motif) |\n| **Running Time** | 148 minutes (standard theatrical cut) |\n| **Budget** | Approximately **$160\u202fmillion** (estimates range from $120\u202fM to $160\u202fM) |\n| **Box‑Office Gross** | **$829.9\u202fmillion** worldwide (as of 2023) |\n| **Genre** | Science‑fiction, Thriller, Action, Psychological Thriller |\n| **Language** | English |\n| **Country of Origin** | United States |\n| **Filming L

In [8]:
model_with_structured_output.invoke("Proved details about the movie Inception")

Movie(title='Inception', year=2010, director='Christopher Nolan', rating=8.8, review='A mind-bending masterpiece that blends cerebral storytelling with spectacular action sequences, showcasing Nolan’s signature style of non-linear narrative and complex visual design. The film’s exploration of reality, memory, and the subconscious is both philosophical and thrilling, with a strong ensemble cast and a haunting score by Hans Zimmer. The layers of dreams within dreams create a rich, immersive experience that rewards multiple viewings, though some viewers may find the plot’s intricacy challenging. Overall, Inception remains a landmark in sci-fi cinema, celebrated for its intellectual depth and technical brilliance.', plot='A thief who steals corporate secrets through use of dream-sharing technology is given the inverse task of planting an idea into the mind of a CEO.', cast=['Leonardo DiCaprio', 'Joseph Gordon-Levitt', 'Elliot Page', 'Tom Hardy', 'Ken Watanabe', 'Cillian Murphy', 'Marion Co

In [9]:
response = model_with_structured_output.invoke("Proved details about the movie Obession 2026")
response

Movie(title='Obsession', year=2026, director='Alexandra Reyes', rating=8.3, review="Obsession masterfully blends cerebral intrigue with visceral tension, showcasing stellar performances and a haunting score that keeps viewers on edge. The film's exploration of memory and identity feels timely, though its pacing may feel uneven for some.", plot='A brilliant neuroscientist becomes obsessed with unlocking the secrets of human memory, leading her to a dangerous experiment that blurs the line between reality and illusion. As her colleagues warn her of the ethical implications, she must confront her own past and decide whether the pursuit of knowledge is worth the cost of her sanity.', cast=['Lena Headey', 'John Boyega', 'Tessa Thompson', 'Riz Ahmed'], genre=['Thriller', 'Psychological'])

In [10]:
response.rating

8.3

In [11]:
response.cast

['Lena Headey', 'John Boyega', 'Tessa Thompson', 'Riz Ahmed']

### Message output alongside parsed structure

In [12]:
from pydantic import BaseModel, Field

class Movie(BaseModel):
    """A movie with details."""
    title: str = Field(..., description="The title of the movie")
    year: int = Field(..., description="The year the movie was released")
    director: str = Field(..., description="The director of the movie")
    rating: float = Field(..., description="The movie's rating out of 10")

model_with_structure = model.with_structured_output(Movie, include_raw=True)  

response = model_with_structure.invoke("Provide details about the movie Inception")
response

{'raw': AIMessage(content='', additional_kwargs={'reasoning_content': "We need to use the provided function to get details. The function signature: Movie({director, rating, title, year}). We need to provide the movie details. So call the function with director: Christopher Nolan, rating: maybe 8.8? title: Inception, year: 2010. But rating might be from IMDB. It's 8.8. We'll use that.", 'tool_calls': [{'id': 'fc_dba6f541-c781-4017-a1d1-6e84abbf4a2c', 'function': {'arguments': '{"director":"Christopher Nolan","rating":8.8,"title":"Inception","year":2010}', 'name': 'Movie'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 120, 'prompt_tokens': 164, 'total_tokens': 284, 'completion_time': 0.132596579, 'completion_tokens_details': {'reasoning_tokens': 81}, 'prompt_time': 0.011799147, 'prompt_tokens_details': None, 'queue_time': 0.355336727, 'total_time': 0.144395726}, 'model_name': 'openai/gpt-oss-20b', 'system_fingerprint': 'fp_1074f9ce08', 'service_tier': 'o

### Nested Structure

In [13]:
from pydantic import BaseModel, Field

class Actor(BaseModel):
    name: str
    role: str

class MovieDetails(BaseModel):
    title: str
    year: int
    cast: list[Actor]
    genres: list[str]
    budget: float | None = Field(None, description="Budget in millions USD")

In [14]:
model_structure= model.with_structured_output(MovieDetails)

In [15]:
response = model_structure.invoke("Provide details about the movie Inception")
response

MovieDetails(title='Inception', year=2010, cast=[Actor(name='Leonardo DiCaprio', role='Dom Cobb'), Actor(name='Joseph Gordon-Levitt', role='Arthur'), Actor(name='Elliot Page', role='Ariadne'), Actor(name='Tom Hardy', role='Eames'), Actor(name='Ken Watanabe', role='Saito')], genres=['Action', 'Adventure', 'Sci-Fi', 'Thriller'], budget=220000000.0)

### TypedDict
TypedDict provides a simpler alternative using Python’s built-in typing, ideal when you don’t need runtime validation.

In [16]:
from typing_extensions import TypedDict,Annotated

class MoveDict(TypedDict):
    """A movie with details """
    title: Annotated[str, "The title of the movie"]
    year: Annotated[int, "The year the movie was released"]
    director: Annotated[str, "The director of the movie"]
    rating: Annotated[float, "The movie's rating out of 10"]

In [17]:
model_struc_dict = model.with_structured_output(MoveDict)

In [18]:
response = model_struc_dict.invoke("Provide details about the movie Avengers")

In [19]:
response

{'title': 'The Avengers'}

In [22]:
class Actor(TypedDict):
    name: str
    role: str

class MovieDetails(TypedDict):
    title: str
    year: int
    cast: list[Actor]
    genres: list[str]
    budget: float | None = Field(None, description="Budget in millions USD")

model_typedDict = model.with_structured_output(MovieDetails)
response = model_typedDict.invoke("Provide details about the movie Avengers")
response
   

{'budget': 220000000,
 'cast': [{'name': 'Robert Downey Jr.', 'role': 'Tony Stark / Iron Man'},
  {'name': 'Chris Evans', 'role': 'Steve Rogers / Captain America'},
  {'name': 'Mark Ruffalo', 'role': 'Bruce Banner / Hulk'},
  {'name': 'Chris Hemsworth', 'role': 'Thor'},
  {'name': 'Scarlett Johansson', 'role': 'Natasha Romanoff / Black Widow'},
  {'name': 'Jeremy Renner', 'role': 'Clint Barton / Hawkeye'},
  {'name': 'Tom Hiddleston', 'role': 'Loki'},
  {'name': 'Samuel L. Jackson', 'role': 'Nick Fury'},
  {'name': 'Clark Gregg', 'role': 'Agent Coulson'}],
 'genres': ['Action', 'Adventure', 'Sci-Fi', 'Superhero'],
 'title': 'Avengers',
 'year': 2012}

In [24]:
response = model_typedDict.invoke("Provide details about the Dexter amazon web series")
response

{'budget': 30000000,
 'cast': [{'name': 'Michael C. Hall', 'role': 'Dexter Morgan'},
  {'name': 'Jennifer Carpenter', 'role': 'Debra Morgan'},
  {'name': 'Julie Benz', 'role': 'Rita Bennett'},
  {'name': 'Yvonne Strahovski', 'role': 'Hannah McKay'}],
 'genres': ['Crime', 'Thriller', 'Drama'],
 'title': 'Dexter',
 'year': 2006}

In [27]:
model.profile

{'name': 'GPT OSS 20B',
 'release_date': '2025-08-05',
 'last_updated': '2026-05-27',
 'open_weights': True,
 'max_input_tokens': 131072,
 'max_output_tokens': 65536,
 'text_inputs': True,
 'image_inputs': False,
 'audio_inputs': False,
 'video_inputs': False,
 'text_outputs': True,
 'image_outputs': False,
 'audio_outputs': False,
 'video_outputs': False,
 'reasoning_output': True,
 'tool_calling': True,
 'structured_output': True,
 'attachment': False,
 'temperature': True}

### DataClasses
A data class is a class typically containing mainly data, although there aren’t really any restrictions. You create it using the @dataclass decorator

In [28]:
import os
os.environ["OPENAI_API_KEY"]=os.getenv("OPENAI_API_KEY")

In [29]:
from pydantic import BaseModel, Field
from langchain.agents import create_agent


class ContactInfo(BaseModel):
    """Contact information for a person."""
    name: str = Field(description="The name of the person")
    email: str = Field(description="The email address of the person")
    phone: str = Field(description="The phone number of the person")

In [30]:
agent = create_agent(
    model="gpt-3.5-turbo",
    response_format=ContactInfo  # Auto-selects ProviderStrategy
)

In [31]:
result = agent.invoke({
    "messages":[
        {
            "role": "user",
            "content":  "Extract contact info from: John Doe, john@example.com, (555) 123-4567"
        }
        ]
})

result

{'messages': [HumanMessage(content='Extract contact info from: John Doe, john@example.com, (555) 123-4567', additional_kwargs={}, response_metadata={}, id='63e429a2-59d3-4d75-8a6a-4cf5d7540d60'),
  AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 32, 'prompt_tokens': 96, 'total_tokens': 128, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0, 'text_tokens': None}, 'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': None, 'cached_tokens': 0, 'image_tokens': None, 'text_tokens': None}}, 'model_provider': 'openai', 'model_name': 'gpt-3.5-turbo-0125', 'system_fingerprint': None, 'id': 'chatcmpl-EUo47p1zaSOqE7Y2jNMQV2DUd3H4W', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--01a10093-060f-7c63-834a-71528318819e-0', tool_calls=[{'name': 'ContactInfo', 'args': {'name': 'John Doe', 'email':

In [32]:
result["structured_response"]

ContactInfo(name='John Doe', email='john@example.com', phone='(555) 123-4567')

In [33]:
## TypedDict
from typing_extensions import TypedDict,Annotated

class ContactInfo(TypedDict):
    """Contact information for a person."""
    name: str = Field(description="The name of the person")
    email: str = Field(description="The email address of the person")
    phone: str = Field(description="The phone number of the person")

agent = create_agent(
    model="gpt-3.5-turbo",
    response_format=ContactInfo  # Auto-selects ProviderStrategy
)

result = agent.invoke({
    "messages":[
        {
            "role": "user",
            "content":  "Extract contact info from: John Doe, john@example.com, (555) 123-4567"
        }
        ]
})

result

{'messages': [HumanMessage(content='Extract contact info from: John Doe, john@example.com, (555) 123-4567', additional_kwargs={}, response_metadata={}, id='8143b220-2f71-47c8-8ee6-71f4cd5674e7'),
  AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 47, 'prompt_tokens': 72, 'total_tokens': 119, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0, 'text_tokens': None}, 'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': None, 'cached_tokens': 0, 'image_tokens': None, 'text_tokens': None}}, 'model_provider': 'openai', 'model_name': 'gpt-3.5-turbo-0125', 'system_fingerprint': None, 'id': 'chatcmpl-EUo7LxOm6PHAT4bX9LcLpBnITw6J0', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--01a10096-1610-7693-be2b-d567e5776204-0', tool_calls=[{'name': 'ContactInfo', 'args': {'name': 'John Doe', 'email':

In [34]:
result["structured_response"]

{'name': 'John Doe', 'email': 'john@example.com', 'phone': '(555) 123-4567'}

In [35]:
## DataClass

from dataclasses import dataclass
from langchain.agents import create_agent


@dataclass
class ContactInfo:
    """Contact information for a person."""
    name: str = Field(description="The name of the person")
    email: str = Field(description="The email address of the person")
    phone: str = Field(description="The phone number of the person")

agent = create_agent(
    model="gpt-3.5-turbo",
    response_format=ContactInfo  # Auto-selects ProviderStrategy
)

result = agent.invoke({
    "messages":[
        {
            "role": "user",
            "content":  "Extract contact info from: John Doe, john@example.com, (555) 123-4567"
        }
        ]
})

result["structured_response"]


ContactInfo(name='John Doe', email='john@example.com', phone='(555) 123-4567')